# PyKOA `send_sync` fix — interactive test notebook

**What this notebook is:**  
A human-readable walkthrough of the five bugs fixed in the `fix/send-sync` branch, with one runnable cell per fix.
It exercises exactly the same code paths as the 26-test pytest suite, and then makes **live calls to the real KOA TAP service** at the end.

**How to use it:**  
Run cells **top to bottom**. Each cell prints `✅` if the check passed. The live-KOA cells at the end need a network connection.

---

## Background — why `send_sync` never worked

`KoaTap.send_sync` is the synchronous query method of PyKOA's TAP client. It has existed in the codebase since 2019 but has **never successfully returned a result**. Five bugs stacked on top of each other:

| # | Bug | Symptom |
|---|---|---|
| 1 | `requesrs.post` — typo for `requests.post` | `NameError: name 'requesrs' is not defined` |
| 2 | `allow_redicts=False` — typo for `allow_redirects` | `TypeError` once bug 1 is fixed |
| 3 | `self.response_result` never set by `send_sync` | `AttributeError: NoneType … iter_content` |
| 4 | No non-200 check — server errors saved to disk as success | HTTP 403 quietly saved as output file |
| 5 | `Table.read(format='votable')` hard-coded | `IORegistryError` / `ValueError` for ipac/csv/tsv |

The fixes are all in `pykoa/koa/core.py` (`KoaTap.send_sync` and `KoaTap.save_data`). The new test file is `tests/test_send_sync.py`.

---

## Step 0 — make sure the kernel uses the branch code

`pip install -e .` installs pykoa from this repo into the Jupyter kernel in 'editable' mode,
so `import pykoa` loads the branch code (with the fixes) instead of whatever version was installed previously.

After running this cell you should see the path to `core.py` inside this repo.
If the kernel was already running with an older version, restart it after this cell.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout); print(r.stderr)
else:
    print('✅ pykoa installed from repo')

# Reload to pick up the branch version
import importlib
import pykoa.koa.core as c; importlib.reload(c)
print('core.py in use:', c.__file__)
print('Expect: …/pykoa-clean/pykoa/koa/core.py')

## Step 1 — shared helpers (run once)

This cell sets up everything the test cells need:

- **`fake_server(status, content_type, body)`** — patches `requests.adapters.HTTPAdapter.send` so no real network call is made. The real `requests.Session.request` code still runs (so both typos cause real errors at the right layer), but the response comes from our fake instead of KOA.
- **`isolated_cwd()`** — patches `tempfile.mkstemp` so any temp file `save_data` creates goes into a private directory we control. This lets us assert 'no temp file was leaked' without moving the process's working directory.
- **`votable_bytes()`, `format_bytes(fmt)`, `error_votable(msg)`** — build small fake server response bodies in the right formats.

No assertions here — just definitions.

In [ ]:
import io, os, sys, tempfile, contextlib
from unittest import mock
from urllib.parse import parse_qs

import requests
from astropy.table import Table
from requests.adapters import HTTPAdapter
from requests.structures import CaseInsensitiveDict

sys.path.insert(0, '.')
from pykoa.koa.core import KoaTap
import pykoa.koa.core as _core

TAP_URL   = 'https://koa.example.test/TAP'   # fake endpoint — no real traffic
QUERY     = 'select top 1 koaid from koa_hires'
KOAID     = 'HI.20200101.12345.fits'          # expected row in fake VOTable responses
TABLE_ERR = ("Table 'nosuchtable' is not available for querying. "
             "Use TAP_SCHEMA.tables to see available tables.")

def votable_bytes():
    """One-row VOTable body — what KOA returns for a successful votable query."""
    buf = io.BytesIO()
    Table({'koaid': [KOAID]}).write(buf, format='votable')
    return buf.getvalue()

def format_bytes(fmt):
    """One-row body in the given TAP format (votable/ipac/csv/tsv)."""
    apy = {'votable':'votable','ipac':'ascii.ipac','csv':'ascii.csv','tsv':'ascii.tab'}[fmt]
    fd, fpath = tempfile.mkstemp()
    os.close(fd)
    Table({'koaid':[KOAID],'instrume':['HIRES']}).write(fpath, format=apy, overwrite=True)
    with open(fpath,'rb') as fh: data = fh.read()
    os.unlink(fpath)
    return data

def error_votable(msg):
    """nexsciTAP-shaped error VOTable — what KOA returns for 400/403 queries."""
    return ('<?xml version="1.0" encoding="UTF-8"?>\n'
            '<VOTABLE version="1.4" xmlns="http://www.ivoa.net/xml/VOTable/v1.3">\n'
            '<RESOURCE type="results">\n'
            '<INFO name="QUERY_STATUS" value="ERROR">\n'
            f'{msg}\n</INFO>\n</RESOURCE>\n</VOTABLE>\n').encode('utf-8')

def fake_server(status=200, content_type='text/xml', body=None):
    """Context manager: intercepts HTTPAdapter.send and returns a fake response.
    Real requests.Session.request code still runs — both typo fixes are exercised.
    """
    if body is None: body = votable_bytes()
    def send(adapter, prepared, **kw):
        r = requests.Response()
        r.status_code = status
        r.headers = CaseInsensitiveDict({'Content-type': content_type})
        r.encoding = requests.utils.get_encoding_from_headers(r.headers)
        r.raw = io.BytesIO(body)
        r.url, r.request = prepared.url, prepared
        return r
    return mock.patch.object(HTTPAdapter, 'send', autospec=True, side_effect=send)

@contextlib.contextmanager
def isolated_cwd():
    """Redirect save_data's temp file into a private directory we can inspect.
    save_data calls tempfile.mkstemp(dir='./') so we patch mkstemp to redirect it.
    """
    with tempfile.TemporaryDirectory() as td:
        orig = tempfile.mkstemp
        def patched(suffix='', prefix='tmp', dir=None, text=False):
            return orig(suffix=suffix, prefix=prefix, dir=td, text=text)
        with mock.patch('pykoa.koa.core.tempfile.mkstemp', side_effect=patched):
            yield td

print('✅ helpers ready')

---
## Fix 1 & 2 — the two typos (`requesrs`, `allow_redicts`)

**What was broken:**  
The no-cookie branch of `send_sync` (`core.py:5258`) called `requesrs.post(…, allow_redicts=False)`.  
- Bug 1: `requesrs` → `NameError` immediately.  
- Bug 2: `allow_redicts` → `TypeError` from `requests.Session.request` once bug 1 was fixed.

**What the test checks:**  
A successful `send_sync` call without a cookie file. If either typo were still present, the call would raise inside the real `requests` stack before the fake server is ever reached. The test passes only if `requests.post` is called with `allow_redirects=False` (the correct spelling).

**What you should see:** `✅ Fix 1 & 2: requests.post called, allow_redirects accepted`

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    out = os.path.join(tmp, 'result.xml')
    with fake_server():
        msg = KoaTap(TAP_URL).send_sync(QUERY, format='votable', outpath=out)
    assert msg == f'Result downloaded to file [{out}]', f'Unexpected message: {msg!r}'
    assert os.path.exists(out), 'Output file was not written'
print('✅ Fix 1 & 2: requests.post called, allow_redirects accepted')

---
## Fix 3 — `self.response_result` was never set by `send_sync`

**What was broken:**  
After `requests.post` succeeded, `send_sync` called `save_data()`. `save_data` streams the response body via `self.response_result.iter_content()`. But `send_sync` never set `self.response_result` — only `send_async` did. So `self.response_result` was `None`, giving:  
`AttributeError: 'NoneType' object has no attribute 'iter_content'`

**The fix:** one line added — `self.response_result = self.response` — just before `save_data` is called. This mirrors exactly how `send_async` works at `core.py:5106`.

**What the test checks:**
- **With an output file:** the file is written to disk and the success message is returned. No temp file leaks into the working directory.
- **In memory (no outpath):** the result is read into `tap.astropytbl` as an astropy Table, the temp file that `save_data` created is deleted afterward, and nothing is left behind.

**What you should see:**  
`✅ Fix 3 (with outpath): save_data streamed correctly`  
`✅ Fix 3 (in memory): astropytbl set, temp file cleaned up`

In [ ]:
# With outpath — output file goes to a separate tmp dir; working dir stays clean
with tempfile.TemporaryDirectory() as out_dir:
    with isolated_cwd() as work_dir:
        out = os.path.join(out_dir, 'r.xml')
        with fake_server():
            msg = KoaTap(TAP_URL).send_sync(QUERY, format='votable', outpath=out)
        assert os.path.exists(out), 'Output file was not written'
        assert os.listdir(work_dir) == [], f'Temp file leaked into cwd: {os.listdir(work_dir)}'
print('✅ Fix 3 (with outpath): save_data streamed correctly')

# In memory — save_data creates + removes a temp XML file, then sets astropytbl
with isolated_cwd() as work_dir:
    with fake_server():
        tap = KoaTap(TAP_URL)
        msg = tap.send_sync(QUERY, format='votable')
    assert msg == 'Result saved in memory (astropy table).', f'Unexpected: {msg!r}'
    assert hasattr(tap, 'astropytbl') and tap.astropytbl is not None, 'astropytbl not set'
    assert os.listdir(work_dir) == [], f'Temp file not cleaned up: {os.listdir(work_dir)}'
print('✅ Fix 3 (in memory): astropytbl set, temp file cleaned up')

---
## Fix 4 — non-200 server responses were silently saved as success

**What was broken:**  
If KOA returned HTTP 403 (access denied) or HTTP 400 (bad query), `send_sync` had no check for the status code. It passed the error response body straight to `save_data`, which wrote it to disk and returned `'Result downloaded to file […]'`. The user saw a success message but actually received an error VOTable.

**The fix:** a guard block added before `save_data` (`core.py:5282-5311`). For any non-200 response:
1. Extract KOA's error message — from the VOTable `QUERY_STATUS=ERROR` body for `format=votable`, or from the plain-text body for `format=ipac/csv/tsv`.
2. The content type is lowercased before matching (so `TEXT/XML` works too).
3. For XML bodies, `.content` (raw bytes) is used instead of `.text` so the XML declaration governs encoding — without this, non-ASCII error text is garbled via Latin-1.
4. Set `tap.status = 'error'` and `tap.msg` **before** calling `response.close()`, so a close failure can't hide the error.
5. Return the error message string. Nothing is written to disk.

**What the test checks (5 scenarios):**
- 403 with a VOTable error body → KOA's exact error message returned
- 400 with a `text/plain` error body → stripped body returned
- 500 with an HTML body (unknown type) → generic `'Request failed with HTTP status 500.'`
- 403 with a non-ASCII VOTable → correct Unicode text (not garbled Latin-1)
- 403 with uppercase `TEXT/XML` header → still matched correctly via `.lower()`

For all five: `tap.status == 'error'`, no output file, no temp file left behind.

**What you should see:** five `✅ Fix 4 (…)` lines.

In [ ]:
cases = [
    # (description, HTTP status, Content-Type, response body, expected message)
    ('403 VOTable — KOA table access denied',
     403, 'text/xml', error_votable(TABLE_ERR), TABLE_ERR),

    ('400 text/plain — bad ADQL column name (plain-text error for non-votable formats)',
     400, 'text/plain; charset=UTF-8', b'Bad column\n', 'Bad column'),

    ('500 HTML — unknown content type falls back to generic message',
     500, 'text/html', b'<html>Internal Server Error</html>',
     'Request failed with HTTP status 500.'),

    ('403 non-ASCII VOTable — verifies .content is used (not .text which would garble via Latin-1)',
     403, 'text/xml', error_votable("Table '\u0152uvre_\xe9' not found"),
     "Table '\u0152uvre_\xe9' not found"),

    ('403 uppercase Content-Type header — verifies .lower() is applied before matching',
     403, 'TEXT/XML', error_votable(TABLE_ERR), TABLE_ERR),
]

with isolated_cwd() as work_dir:
    for desc, status, ctype, body, expected in cases:
        with mock.patch.object(requests.Response, 'close', autospec=True):
            with fake_server(status=status, content_type=ctype, body=body):
                tap = KoaTap(TAP_URL)
                msg = tap.send_sync(QUERY, format='votable')
        assert msg == expected, f'WRONG MESSAGE\n  got:      {msg!r}\n  expected: {expected!r}'
        assert tap.status == 'error', f'tap.status should be "error", got {tap.status!r}'
        assert os.listdir(work_dir) == [], f'Temp file leaked: {os.listdir(work_dir)}'
        print(f'✅ Fix 4 ({desc})')

---
## Fix 5 — `Table.read` had `format='votable'` hard-coded

**What was broken:**  
When `send_sync` is called without an `outpath`, `save_data` writes the server response to a temp file and reads it back into memory with `Table.read(fpath, format='votable')`. The format was hard-coded as `'votable'`.  
But `Archive`'s default format is `'ipac'`, and all Archive query methods advertise four formats: `votable`, `ipac`, `csv`, `tsv`.  
Calling `send_sync(q, format='ipac')` without an outpath would:
- write an IPAC table to the temp file (correct)
- then try to read it as a VOTable → `ValueError` (crash)
- leave the temp file behind in the user's current directory
- `tsv` was even worse: astropy has no reader named `'tsv'` → `IORegistryError` immediately

**The fix:** apply the same TAP→astropy name translation that already exists at `core.py:2891`:  
`ipac → ascii.ipac`, `csv → ascii.csv`, `tsv → ascii.tab`, `votable → votable`

**What the test checks:**  
All four formats, in memory (no outpath). Each one:
- returns `'Result saved in memory (astropy table).'`
- `tap.astropytbl['koaid'][0]` equals the expected KOAID value (the body was decoded correctly)
- no temp file is left in the working directory

**What you should see:** four `✅ Fix 5 (format=…)` lines.

In [ ]:
for fmt in ['votable', 'ipac', 'csv', 'tsv']:
    body = format_bytes(fmt)   # server response in the requested format
    with isolated_cwd() as work_dir:
        with fake_server(body=body):
            tap = KoaTap(TAP_URL)
            msg = tap.send_sync(QUERY, format=fmt)
        assert msg == 'Result saved in memory (astropy table).', \
            f'format={fmt!r}: unexpected message: {msg!r}'
        assert list(tap.astropytbl['koaid']) == [KOAID], \
            f'format={fmt!r}: wrong rows: {list(tap.astropytbl["koaid"])}'
        assert os.listdir(work_dir) == [], \
            f'format={fmt!r}: temp file leaked: {os.listdir(work_dir)}'
    print(f'✅ Fix 5 (format={fmt!r}): {len(tap.astropytbl)} row decoded correctly')

---
## Live KOA checks — real network calls

These cells call `https://koa.ipac.caltech.edu/TAP` directly. They require a network connection and will fail if KOA is down.

**What they check:**
1. **Good query → output file:** 3 real HIRES records are returned, the file parses correctly as a VOTable.
2. **Good query → in memory:** same query, result is in `tap.astropytbl`.
3. **Empty result (OVERFLOW):** `select … where 1=0` returns HTTP 200 with `QUERY_STATUS=OVERFLOW`. The fix treats this as success (0 rows), not an error.
4. **Bad query → error returned:** a query for a nonexistent table returns HTTP 403. The non-200 guard (Fix 4) captures KOA's error message, returns it as a string, sets `tap.status='error'`, and writes nothing to disk.

This is the most important check — it confirms the fixes work against the real service, not just the fake server.

In [ ]:
KOA = 'https://koa.ipac.caltech.edu/TAP'

# 1. Good query — save to file
with tempfile.TemporaryDirectory() as tmp:
    out = os.path.join(tmp, 'result.xml')
    t = KoaTap(KOA)
    msg = t.send_sync('select top 3 koaid from koa_hires', format='votable', outpath=out)
    tbl = Table.read(out, format='votable')
    print(f'✅ Live outpath: HTTP {t.response.status_code}, {len(tbl)} rows returned')
    print(f'   First KOAID: {tbl["koaid"][0]}')

# 2. Good query — in memory
t = KoaTap(KOA)
msg = t.send_sync('select top 3 koaid from koa_hires', format='votable')
print(f'✅ Live in-memory: {len(t.astropytbl)} rows, msg={msg!r}')

# 3. Empty result — QUERY_STATUS=OVERFLOW must be treated as success, not an error
t = KoaTap(KOA)
msg = t.send_sync('select top 1 koaid from koa_hires where 1=0', format='votable')
assert t.status != 'error', f'Empty result wrongly treated as error: {msg!r}'
print(f'✅ Live empty (OVERFLOW): HTTP {t.response.status_code}, {len(t.astropytbl)} rows, not an error')

# 4. Bad query — KOA should return 403 with QUERY_STATUS=ERROR
with isolated_cwd() as work_dir:
    t = KoaTap(KOA)
    msg = t.send_sync('select nosuchcol from nosuchtable', format='votable')
    assert t.status == 'error', f'Bad query should set status=error, got {t.status!r}'
    assert os.listdir(work_dir) == [], f'Error response should not leave temp file: {os.listdir(work_dir)}'
    print(f'✅ Live error: HTTP {t.response.status_code}, status={t.status!r}')
    print(f'   KOA message: {msg}')

---
## Summary

If all cells above showed `✅`, every fix is working correctly:

| Fix | Symptom before | Confirmed by |
|---|---|---|
| 1 & 2 — typos | `NameError` / `TypeError` | Successful call in Fix 1 & 2 cell |
| 3 — `response_result` | `AttributeError` on `iter_content` | File written + astropytbl set in Fix 3 cells |
| 4 — non-200 guard | 403 saved as success | 5 error scenarios in Fix 4 cell |
| 5 — format mapping | `IORegistryError` / `ValueError` for non-votable | All 4 formats pass in Fix 5 cell |

For the full parametrised test suite (26 tests including edge cases) run:
```
python -m pytest tests/test_send_sync.py -v
```
from the repo root.